In [1]:
# Cell 1 — Load raw inputs and audit missing values
# Purpose: document the starting state of each file before any cleaning.
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path.home() / "pavement_index"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"

seg = gpd.read_file(RAW / "2026-10-24sd_paving_segs_datasd.geojson")   # street segments + 2023 PCI
rep = pd.read_csv(RAW / "completed_paving_datasd.csv", low_memory=False) # repair history (many rows per segment)
cpn = pd.read_csv(PROC / "segment_cpname.csv")                           # planning area per segment (Day 1 join)

print(f"segments : {seg.shape[0]:,} rows x {seg.shape[1]} cols | CRS: {seg.crs}")
print(f"repairs  : {rep.shape[0]:,} rows x {rep.shape[1]} cols")
print(f"cpname   : {cpn.shape[0]:,} rows | columns: {list(cpn.columns)}")

# Missing-value audit: count and percent of blanks per column
audit = pd.DataFrame({"n_missing": seg.isna().sum(),
                      "pct_missing": (seg.isna().mean() * 100).round(1)})
print("\nMissing values in the segment layer:")
print(audit.sort_values("pct_missing", ascending=False).to_string())

segments : 32,745 rows x 20 cols | CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]]
repairs  : 40,323 rows x 24 cols
cpname   : 32,745 rows | columns: ['seg_key', 'cpname']

Missing values in the segment layer:
            n_missing  pct_missing
cpname          32745        100.0
funclass        32745        100.0
paveclass       32745        100.0
pci23_d          1285          3.9
pci23            1285          3.9
pv_llowadd         66          0.2
pv_lhighad         66          0.2
pv_rlowadd         66          0.2
pv_rhighad         66          0.2
xstrt2             18          0.1
objectid            0          0.0
lane_no             0          0.0
zip                 2          0.0
pav_length          0          0

In [2]:
# Cell 2 — Select project columns and drop segments without a 2023 PCI label
# Purpose: one row per scored segment, only the fields used downstream.
KEEP = ["iamfloc", "rd20full", "zip", "pwidth", "pav_length", "lane_no",
        "pci23", "pci23_d", "geometry"]

n_start = len(seg)
empty_cols = [c for c in seg.columns if seg[c].isna().all()]   # cpname, funclass, paveclass
print("Dropped (100% empty):", empty_cols)

df = seg[KEEP].copy()
df = df[df["pci23"].notna()].copy()                             # no label -> unusable
print(f"Dropped {n_start - len(df):,} segments with blank PCI -> {len(df):,} remain")

# Remaining gaps in the kept columns, after removing unlabeled segments
print("\nRemaining missing values:")
print(df.drop(columns="geometry").isna().sum()[lambda s: s > 0])

# Check that the ID is still unique (one row per segment)
print("\niamfloc unique:", df["iamfloc"].is_unique)

Dropped (100% empty): ['paveclass', 'funclass', 'cpname']
Dropped 1,285 segments with blank PCI -> 31,460 remain

Remaining missing values:
Series([], dtype: int64)

iamfloc unique: True


In [3]:
# Cell 3 — Range and outlier checks on numeric fields (inspect only, nothing dropped)
NUM = ["pwidth", "pav_length", "lane_no", "pci23"]   # width/length are in feet
print(df[NUM].describe(percentiles=[.01, .5, .99]).T.round(1))

# Plausibility rules for a city street segment
checks = {
    "pwidth <= 0 ft":                (df["pwidth"] <= 0).sum(),
    "pwidth > 100 ft":               (df["pwidth"] > 100).sum(),
    "pav_length < 20 ft":            (df["pav_length"] < 20).sum(),
    "pav_length > 2640 ft (0.5 mi)": (df["pav_length"] > 2640).sum(),
    "lane_no == 0":                  (df["lane_no"] == 0).sum(),
    "pci23 outside 0-100":           (~df["pci23"].between(0, 100)).sum(),
}
print("\nPlausibility flags:")
print(pd.Series(checks, name="n_segments").to_string())

print("\nSegments by number of lanes:")
print(df["lane_no"].value_counts().sort_index().to_string())

              count   mean    std   min    1%    50%     99%      max
pwidth      31460.0   39.0   13.6  10.0  20.0   36.0    82.0    999.0
pav_length  31460.0  499.4  478.1   8.0  81.0  372.5  2183.8  18791.1
lane_no     31460.0    2.2    0.6   0.0   2.0    2.0     5.0      8.0
pci23       31460.0   66.2   25.1   0.0   7.6   73.7   100.0    100.0

Plausibility flags:
pwidth <= 0 ft                     0
pwidth > 100 ft                   60
pav_length < 20 ft                 5
pav_length > 2640 ft (0.5 mi)    208
lane_no == 0                       5
pci23 outside 0-100                0

Segments by number of lanes:
lane_no
0        5
1      278
2    28490
3      447
4     1858
5       98
6      268
7       13
8        3


In [4]:
# Cell 4 — Inspect suspicious values, then convert placeholder codes to missing
import numpy as np

# Inspect: which widths exceed 100 ft?
print("pwidth values > 100 ft (value: count):")
print(df.loc[df["pwidth"] > 100, "pwidth"].value_counts().sort_index().to_string())
print("\nSegments with lane_no == 0:")
print(df.loc[df["lane_no"] == 0, ["rd20full", "pwidth", "pav_length", "pci23"]].to_string())

# Fix: placeholder codes -> NaN (keep the row; its PCI label is still valid)
df["pwidth"] = df["pwidth"].replace(999, np.nan)
df.loc[df["lane_no"] == 0, "lane_no"] = np.nan

# Flags used later (analysis sets and plot checks)
df["is_residential"] = (df["lane_no"] <= 2).astype("Int64").where(df["lane_no"].notna())
df["long_segment"] = (df["pav_length"] > 2640).astype(int)

print("\nAfter fix — missing:", df[["pwidth", "lane_no"]].isna().sum().to_dict())
print("Residential share:", round(df["is_residential"].mean() * 100, 1), "%")

pwidth values > 100 ft (value: count):
pwidth
101.0     2
102.0    27
105.0     2
106.0     7
108.0     1
110.0     3
111.0     1
112.0     3
115.0     1
118.0     5
119.0     2
120.0     1
125.0     1
130.0     3
999.0     1

Segments with lane_no == 0:
           rd20full  pwidth  pav_length  pci23
11551   SHERIDAN LN    40.0    574.5398  26.89
11555   SHERIDAN LN    40.0    579.5585  46.08
14968  GOLDFINCH PL    20.0    548.0011  33.83
26048   WHITTIER ST    50.0    616.0599  29.56
27179     PUEBLO ST    29.0    455.3688  53.76

After fix — missing: {'pwidth': 1, 'lane_no': 5}
Residential share: 91.5 %


### Checkpoint 1 — Initial audit and basic cleaning

- **Starting point:** 32,745 street segments from the City's Streets Repair Line Segments layer
  (downloaded 2026-10-24), stored in geographic coordinates (WGS 84, lon/lat).
- **Empty columns:** planning area, functional class, and pavement class are blank in every row of
  the City export. Planning area was rebuilt by a spatial join (Day 1); the other two are unavailable.
- **Unlabeled segments:** 1,285 segments (3.9%) have no 2023 PCI (unpaved, under construction, or
  obstructed during the survey) and were removed, leaving **31,460 labeled segments**, one row each.
  The few missing ZIP and width values all belonged to these unlabeled segments.
- **Placeholder codes:** one segment recorded a width of 999 ft and five recorded 0 lanes; both are
  "unknown" codes, not measurements, and were set to missing. Rows were kept because their PCI is valid.
  Missing values are filled inside each cross-validation fold at modeling time, not here, to avoid leakage.
- **Plausible extremes kept:** widths of 101–130 ft (wide arterials) and 208 segments longer than
  half a mile (flagged as `long_segment`).
- **Street type:** 91.5% of segments have one or two lanes (`is_residential`), the project's primary
  analysis set.
- **Label shape:** PCI mean 66.2 vs. median 73.7, so the distribution is left-skewed: most streets are
  in good shape, with a long tail of badly deteriorated ones.

In [5]:
# Cell 5 — Repair history -> one row per segment: stale flag + pavement-age features
SURVEY_START = pd.Timestamp("2023-02-01")   # City survey ran Feb–Aug 2023

r = rep[["segment_id", "date_end", "project_type", "status"]].copy()
r["date_end"] = pd.to_datetime(r["date_end"], errors="coerce")
print("Repair rows:", f"{len(r):,}", "| blank/unparseable date_end:", r["date_end"].isna().sum())
print("\nRepair status values:")
print(r["status"].value_counts(dropna=False).to_string())

# Latest repair of any date -> stale flag
last_any = r.groupby("segment_id")["date_end"].max().rename("last_repair_end")
# Latest repair BEFORE the survey -> pavement age as of survey start
last_pre = (r[r["date_end"] < SURVEY_START]
            .groupby("segment_id")["date_end"].max().rename("last_repair_pre_survey"))

df = df.merge(last_any, left_on="iamfloc", right_index=True, how="left")
df = df.merge(last_pre, left_on="iamfloc", right_index=True, how="left")

df["is_stale"] = (df["last_repair_end"] >= SURVEY_START).astype(int)
df["has_prior_repair"] = df["last_repair_pre_survey"].notna().astype(int)
df["years_since_repair"] = ((SURVEY_START - df["last_repair_pre_survey"]).dt.days / 365.25).round(2)

print(f"\nRows still one per segment: {df['iamfloc'].is_unique} ({len(df):,})")
print(f"Stale (repaired since survey start): {df['is_stale'].sum():,} ({df['is_stale'].mean():.1%})")
print(f"Has a pre-survey repair: {df['has_prior_repair'].mean():.1%}")
print("years_since_repair:", df["years_since_repair"].describe().round(1).to_dict())

Repair rows: 40,323 | blank/unparseable date_end: 0

Repair status values:
status
POST CONSTRUCTION    40323

Rows still one per segment: True (31,460)
Stale (repaired since survey start): 8,432 (26.8%)
Has a pre-survey repair: 65.8%
years_since_repair: {'count': 20706.0, 'mean': 6.0, 'std': 4.8, 'min': 0.0, '25%': 3.1, '50%': 5.0, '75%': 7.1, 'max': 27.1}


In [6]:
# Cell 6 — Attach planning area, define the clean training set, save the model-ready table
df = df.merge(cpn.rename(columns={"seg_key": "iamfloc"}), on="iamfloc", how="left")
print(f"Planning area filled: {df['cpname'].notna().mean():.1%} "
      f"({df['cpname'].nunique()} areas) | rows: {len(df):,}")

# Clean set = labeled AND not repaired since the survey began
df["in_clean_set"] = (df["is_stale"] == 0).astype(int)
clean = df[df["in_clean_set"] == 1]
print(f"Clean (not stale): {len(clean):,} | of which residential: "
      f"{(clean['is_residential'] == 1).sum():,}")

# Stale share by PCI band (supports the 'labels age' finding in the report)
band_order = ["Good", "Satisfactory", "Fair", "Poor", "Very Poor", "Serious", "Failed"]
stale_by_band = (df.groupby("pci23_d")["is_stale"].mean().mul(100).round(1)
                   .reindex(band_order))
print("\nStale % by PCI band:\n", stale_by_band.to_string())

# Save (no geometry; maps re-load it from the raw GeoJSON by iamfloc)
OUT = PROC / "segments_clean.csv"
df.drop(columns="geometry").to_csv(OUT, index=False)
print(f"\nSaved {OUT.name}: {len(df):,} rows x {df.shape[1] - 1} cols")

Planning area filled: 99.9% (57 areas) | rows: 31,460
Clean (not stale): 23,028 | of which residential: 21,002

Stale % by PCI band:
 pci23_d
Good            19.9
Satisfactory    36.4
Fair            30.7
Poor            23.4
Very Poor       17.0
Serious         19.1
Failed          32.9

Saved segments_clean.csv: 31,460 rows x 17 cols


### Checkpoint 2 — Repair history, planning area, and the clean set

- **Repair history:** 40,323 completed-project records (all "Post Construction", all dated), several
  per segment, were reduced to one row per segment before joining.
- **Stale labels:** 8,432 segments (26.8%) had a repair finish on or after 2023-02-01, the start of the
  City survey, so their 2023 PCI may describe the street before treatment. They are excluded from
  training and kept as a separate check (H3). Staleness is highest in the middle bands (Satisfactory
  36.4%, Fair 30.7%), consistent with preventive treatment of moderately worn streets.
- **Pavement age:** 65.8% of segments had a repair before the survey; for those, the median time since
  that repair was 5.0 years (max 27.1). Segments with no record are left blank rather than set to 0,
  because "no record" is not "just repaved." Age is measured as of the survey start, so no
  post-survey information enters the feature.
- **Planning area:** joined for 99.9% of segments across 57 areas; used for leave-area-out testing.
- **Output:** `data/processed/segments_clean.csv` (31,460 rows). The clean training pool is
  **23,028 segments, 21,002 residential**. The model only ever sees the subset that is actually driven.